In [ ]:
# ---------------------- 1. 数据处理 ----------------------
# 导入数据处理库
import pandas as pd  # 处理表格数据，读取Excel文件
import numpy as np   # 处理数值计算和数组操作
# 导入深度学习库
import tensorflow as tf  # TensorFlow深度学习框架
from tensorflow.keras import layers, Model  # 用于构建模型的层和模型类
from tensorflow.keras.optimizers import AdamW  # AdamW优化器，提升训练稳定性
from tensorflow.keras.losses import SparseCategoricalFocalCrossentropy  # 解决类别不平衡的损失函数
# 导入机器学习工具
from sklearn.model_selection import train_test_split  # 划分训练集和测试集
from sklearn.preprocessing import LabelEncoder, StandardScaler  # 标签编码和特征标准化
from sklearn.metrics import classification_report, confusion_matrix, cohen_kappa_score  # 模型评估指标
# 导入其他工具
import joblib  # 保存和加载模型、编码器
import rasterio  # 读取遥感影像文件
from rasterio.plot import show  # 可视化遥感影像
import matplotlib.pyplot as plt  # 绘制图表和可视化
import os  # 处理文件路径和目录

In [ ]:
# ---------------------- 2. 数据路径设置 ----------------------

# 数据路径设置，实际使用时替换为你的真实数据路径
# 遥感影像路径
SENTINEL2_PATH = "./data/sentinel2.tif"  # Sentinel-2多光谱影像，10m分辨率，提供光谱信息
SENTINEL1_PATH = "./data/sentinel1.tif"  # Sentinel-1 SAR雷达影像，不受天气影响，提供结构信息
GF2_PATH = "./data/gf2.tif"  # 高分二号高分辨率影像，2m分辨率，提取树冠细节纹理
# 属性数据路径
SPECIES_DATA_PATH = "./data/GUTS_dataset.xlsx"  #全球城市树种数据集
# 结果保存路径
SAVE_PATH = "./output"
# 创建保存目录，如果目录不存在则自动创建
os.makedirs(SAVE_PATH, exist_ok=True)

In [ ]:
# ---------------------- 3. 属性数据预处理部分 ----------------------

# 读取属性数据，Excel格式的树种数据
df = pd.read_excel(SPECIES_DATA_PATH)

# 1. 删除无用特征：这些特征对分类没有帮助，或者是固定值
drop_columns = ['City', 'Country', 'ISO2', 'GeoLat_City', 'GeoLon_City', 'GHS_UrbanAreas', 'GUB_UrbanAreas', 'BIOME', 'WFO_ID']
# errors='ignore'表示如果列不存在就忽略，避免报错
df = df.drop(columns=drop_columns, errors='ignore')

# 2. 缺失值处理：填充缺失的特征值
# 验证类特征：缺失值填充为0，表示未进行该类验证
numeric_cols = ['LiteratureData_UrbanAreas', 'LiteratureData_Administrative', 'GRIIS_verification', 'WCVP_verification', 'GloNAF_verification', 'Manually_confirmed', 'Uncertain', 'Potentially erroneous/Misidentified']
df[numeric_cols] = df[numeric_cols].fillna(0)
# 本地物种标识：缺失值填充为众数1.0（1表示本地物种，0表示外来物种）
df['NativeFlag'] = df['NativeFlag'].fillna(1.0)

# 3. 处理文献编码特征：将分号分隔的多值编码转换为独热编码
# 先将缺失值转换为空字符串
df['Literature_codes'] = df['Literature_codes'].astype(str).replace('nan', '')
# 收集所有的文献编码，用于创建独热编码列
all_codes = set()
for codes in df['Literature_codes']:
    if codes:  # 如果编码不为空
        for code in codes.split('; '):  # 按分号分割每个编码
            all_codes.add(code)
# 转换为排序后的列表，保证每次运行的顺序一致
all_codes = sorted(list(all_codes))
# 为每个编码创建独热编码列，1表示该样本包含这个编码，0表示不包含
for code in all_codes:
    df[f'code_{code}'] = df['Literature_codes'].apply(lambda x: 1 if code in x.split('; ') else 0)
# 删除原始的编码列，因为已经转换为独热编码
df = df.drop(columns=['Literature_codes'])

# 4. 标签编码：将树种名称转换为数值，方便模型训练
le = LabelEncoder()
# 对树种名称进行编码，将字符串转换为整数
df['label'] = le.fit_transform(df['Species_binomial'])
# 保存编码器，用于后续将数值转换回树种名称
joblib.dump(le, os.path.join(SAVE_PATH, 'label_encoder.pkl'))

# 5. 划分特征和标签
# 属性特征：删除树种名称和标签列，剩下的都是用于分类的特征
X_attr = df.drop(columns=['Species_binomial', 'label'])
# 标签：编码后的树种数值，是模型要预测的目标
y = df['label']

# 6. 标准化属性特征：消除量纲影响，提升模型训练稳定性
scaler = StandardScaler()
# 对特征进行标准化，将特征转换为均值为0，标准差为1的分布
X_attr_scaled = scaler.fit_transform(X_attr)
# 保存标准化器，用于后续对新数据进行同样的标准化处理
joblib.dump(scaler, os.path.join(SAVE_PATH, 'attr_scaler.pkl'))

In [ ]:
# ---------------------- 4. 遥感影像预处理部分 ----------------------

# 读取遥感影像的函数，将影像转换为模型需要的格式
def load_remote_image(path):
    with rasterio.open(path) as src:
        # 读取影像，rasterio读取的格式是 (通道数, 高度, 宽度)
        img = src.read()
        # 转换为 (高度, 宽度, 通道数) 格式，符合TensorFlow的输入要求
        img = np.transpose(img, (1, 2, 0))
        # 归一化到[0,1]区间，提升模型训练稳定性，避免数值过大导致训练不稳定
        img = img / np.max(img)
        return img

# 加载多源遥感影像
sentinel2_img = load_remote_image(SENTINEL2_PATH)  # 光学影像，提供光谱信息
sentinel1_img = load_remote_image(SENTINEL1_PATH)  # 雷达影像，弥补光学影像的天气限制
gf2_img = load_remote_image(GF2_PATH)  # 高分辨率影像，提供树冠细节信息

# 裁剪影像块的函数：根据树种的坐标裁剪对应大小的影像块
def crop_image(img, coords, patch_size=64):
    # coords是地理坐标，这里模拟裁剪，实际需要将地理坐标转换为影像的像素坐标
    # img.shape: (高度, 宽度, 通道数)
    H, W, C = img.shape
    patches = []
    # 为每个样本裁剪一个固定大小的影像块
    for _ in range(len(coords)):
        # 随机选择裁剪的位置，实际使用时需要根据地理坐标计算对应的像素坐标
        x = np.random.randint(0, W - patch_size)
        y = np.random.randint(0, H - patch_size)
        # 裁剪patch_size x patch_size的影像块
        patch = img[y:y+patch_size, x:x+patch_size, :]
        patches.append(patch)
    # 转换为numpy数组，方便模型处理
    return np.array(patches)

# 模拟坐标：实际使用时，这里应该是数据集中的GeoLat_City和GeoLon_City
# 这里用数据集中的坐标列，实际需要将地理坐标转换为影像的像素坐标
coords = df[['GeoLat_City', 'GeoLon_City']].values
# 裁剪多源影像的影像块，每个样本对应一个影像块
sentinel2_patches = crop_image(sentinel2_img, coords, patch_size=64)
sentinel1_patches = crop_image(sentinel1_img, coords, patch_size=64)
gf2_patches = crop_image(gf2_img, coords, patch_size=64)

# 融合多源影像：将三个影像的特征拼接在一起，形成多模态的影像特征
X_remote = np.concatenate([sentinel2_patches, sentinel1_patches, gf2_patches], axis=-1)

# 划分训练集和测试集：按9:1的比例划分，stratify=y保证每个树种在训练集和测试集中的分布一致
X_remote_train, X_remote_test, X_attr_train, X_attr_test, y_train, y_test = train_test_split(
    X_remote, X_attr_scaled, y, test_size=0.1, random_state=42, stratify=y
)

In [ ]:
# ---------------------- 5. 模型构建部分 ----------------------

# 1. 遥感影像特征分支：使用Swin Transformer提取影像的空间和光谱特征
def swin_transformer_branch(input_shape):
    # 输入层：影像块的形状，比如(64,64, 通道数)
    inputs = layers.Input(shape=input_shape)
    # 使用SwinV2Small作为骨干网络，不使用预训练权重（因为遥感影像和自然图像的分布不同）
    base_model = tf.keras.applications.SwinV2Small(input_shape=input_shape, include_top=False, weights=None)
    # 提取影像特征
    x = base_model(inputs)
    # 全局平均池化，将特征图转换为一维特征向量
    x = layers.GlobalAveragePooling2D()(x)
    # 输出128维的特征向量，用于后续融合
    outputs = layers.Dense(128, activation='relu')(x)
    return Model(inputs, outputs, name='swin_branch')

# 2. 属性数据特征分支：使用MLP提取属性特征
def mlp_attr_branch(input_shape):
    # 输入层：属性特征的维度
    inputs = layers.Input(shape=input_shape)
    # 第一层全连接层，64个神经元，ReLU激活
    x = layers.Dense(64, activation='relu')(inputs)
    # Dropout层，随机失活20%的神经元，防止过拟合
    x = layers.Dropout(0.2)(x)
    # 第二层全连接层，64个神经元，ReLU激活
    x = layers.Dense(64, activation='relu')(x)
    # 输出128维的特征向量，用于后续融合
    outputs = layers.Dense(128, activation='relu')(x)
    return Model(inputs, outputs, name='attr_branch')

# 3. 交叉注意力融合模块：融合影像特征和属性特征，动态调整两个模态的权重
def cross_attention_fusion(feat_remote, feat_attr):
    # 将特征转换为 (batch, 1, 128) 的形状，适配注意力机制的输入要求
    feat_remote = layers.Reshape((1, 128))(feat_remote)
    feat_attr = layers.Reshape((1, 128))(feat_attr)
    # 交叉注意力：让影像特征关注属性数据的关键信息，同时让属性特征响应影像的关键区域
    attention = layers.Attention()([feat_remote, feat_attr])
    # 拼接原始影像特征和注意力特征，融合更多信息
    fused = layers.Concatenate()([feat_remote, attention])
    # 展平为一维向量
    fused = layers.Flatten()(fused)
    # 再通过一个全连接层，进一步融合特征
    fused = layers.Dense(128, activation='relu')(fused)
    return fused

# 4. 构建完整的多模态融合模型
# 影像输入层：影像块的形状
input_remote = layers.Input(shape=X_remote_train.shape[1:])
# 属性输入层：属性特征的维度
input_attr = layers.Input(shape=X_attr_train.shape[1:])

# 提取影像特征
remote_feat = swin_transformer_branch(X_remote_train.shape[1:])(input_remote)
# 提取属性特征
attr_feat = mlp_attr_branch(X_attr_train.shape[1:])(input_attr)

# 融合两个模态的特征
fused_feat = cross_attention_fusion(remote_feat, attr_feat)
# 分类输出层：输出树种类别的概率，数量是树种的类别数
outputs = layers.Dense(len(le.classes_), activation='softmax')(fused_feat)

# 构建完整模型
model = Model(inputs=[input_remote, input_attr], outputs=outputs)
# 打印模型结构，查看各层的参数和输入输出形状
model.summary()

In [ ]:
# ---------------------- 6. 模型训练部分 ----------------------

# 编译模型，设置优化器、损失函数和评估指标
model.compile(
    optimizer=AdamW(learning_rate=0.001),  # 使用AdamW优化器，学习率0.001
    # 使用SparseCategoricalFocalCrossentropy损失函数，解决类别不平衡问题
    # from_logits=False表示输出是经过Softmax激活的概率
    loss=SparseCategoricalFocalCrossentropy(from_logits=False),
    metrics=['accuracy']  # 评估指标：准确率
)

# 训练模型
history = model.fit(
    # 输入是两个模态的特征：影像特征和属性特征
    [X_remote_train, X_attr_train], y_train,
    epochs=50,  # 训练50个轮次
    batch_size=8,  # 批次大小8，每个批次训练8个样本
    validation_split=0.1,  # 用训练集的10%作为验证集，监控训练过程
    callbacks=[
        # 保存最佳模型，只保存验证集损失最小的模型
        tf.keras.callbacks.ModelCheckpoint(os.path.join(SAVE_PATH, 'best_model.h5'), save_best_only=True),
        # 学习率调度：当验证集损失5个轮次没有下降时，学习率降低为原来的0.5
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)
    ]
)

# 保存训练历史，用于后续可视化训练过程的准确率和损失变化
import json
with open(os.path.join(SAVE_PATH, 'training_history.json'), 'w') as f:
    json.dump(history.history, f)

In [ ]:
# ---------------------- 7. 模型评估部分 ----------------------

# 加载最佳模型，使用验证集损失最小的模型进行评估
best_model = tf.keras.models.load_model(os.path.join(SAVE_PATH, 'best_model.h5'))

# 评估模型在测试集上的表现，得到测试集的损失和准确率
test_loss, test_acc = best_model.evaluate([X_remote_test, X_attr_test], y_test)
print(f"测试集准确率: {test_acc:.4f}")

# 预测测试集的结果，得到每个样本的预测概率
y_pred = best_model.predict([X_remote_test, X_attr_test])
# 将概率转换为预测的标签，取概率最大的类别
y_pred = np.argmax(y_pred, axis=1)

# 生成分类报告，包含每个树种的精确率、召回率、F1-score
report = classification_report(y_test, y_pred, target_names=le.classes_)
print("分类报告:")
print(report)

# 计算Kappa系数，衡量分类结果与真实结果的一致性，Kappa越接近1，一致性越好
kappa = cohen_kappa_score(y_test, y_pred)
print(f"Kappa系数: {kappa:.4f}")

# 绘制混淆矩阵，可视化分类结果
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(12, 12))
# 用热力图展示混淆矩阵，颜色越深表示样本数越多
plt.imshow(cm, cmap='Blues')
plt.title('混淆矩阵')
plt.colorbar()
plt.xlabel('预测标签')
plt.ylabel('真实标签')
# 保存混淆矩阵图到输出目录
plt.savefig(os.path.join(SAVE_PATH, 'confusion_matrix.png'))
# 关闭绘图，避免内存占用
plt.close()

In [ ]:
# ---------------------- 8. 预测示例部分 ----------------------

# 单个样本预测函数，输入模型、影像块和属性数据，返回预测的树种名称和概率
def predict_single_sample(model, remote_patch, attr_data):
    # 增加维度，适配模型的输入要求（模型需要batch维度，即(1, 64,64, 通道数)）
    remote_patch = np.expand_dims(remote_patch, axis=0)
    attr_data = np.expand_dims(attr_data, axis=0)
    # 预测概率
    pred_proba = model.predict([remote_patch, attr_data])[0]
    # 取概率最大的类别作为预测结果
    pred_label = np.argmax(pred_proba)
    # 将数值标签转换为树种名称
    return le.inverse_transform([pred_label])[0], pred_proba[pred_label]

# 取测试集中的第一个样本进行预测
sample_remote = X_remote_test[0]
sample_attr = X_attr_test[0]
# 真实的树种名称
true_species = le.inverse_transform([y_test.iloc[0]])[0]

# 预测
pred_species, pred_prob = predict_single_sample(best_model, sample_remote, sample_attr)
# 打印预测结果
print(f"真实树种: {true_species}")
print(f"预测树种: {pred_species}")
print(f"预测概率: {pred_prob:.4f}")